# Start

In [37]:
# Taruh ini di sel paling atas sendiri di Notebook-mu
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
%%writefile ./src/__init__.py
# 

Overwriting ./src/__init__.py


## sensor_simulation.py

In [39]:
%%writefile ./src/sensor_simulation.py
from datetime import datetime, timezone, timedelta
import random

def sensor_simulation(dummy_size_kb: int = 1) -> dict[str,float]:
  tz_wib = timezone(timedelta(hours=7))
  temperature = random.uniform(20.0, 40.0)
  air_humidity = random.uniform(40.0, 100.0)
  soil_moisture = random.uniform(0.0, 100.0)
  soil_ph = random.uniform(4.0, 7.0)

  return {
      'reading_timestamp': datetime.now(tz_wib).isoformat(),
      'temperature': temperature,
      'air_humidity': air_humidity,
      'soil_moisture': soil_moisture,
      'soil_ph': soil_ph,
      'dummy' : 'x' * 1024 * dummy_size_kb

      ## if prefer lower decimal count to save space
      # round('temperature': temperature, 3),
      # round('air_humidity': air_humidity, 3),
      # round('soil_moisture': soil_moisture, 3),
      # round('soil_ph': soil_ph, 3)
  }

Overwriting ./src/sensor_simulation.py


In [40]:
from sensor_simulation import sensor_simulation

print(sensor_simulation())

{'reading_timestamp': '2026-06-07T15:28:35.550623+07:00', 'temperature': 27.626813892566034, 'air_humidity': 46.58273078909894, 'soil_moisture': 52.31344687963727, 'soil_ph': 5.911098530022178, 'dummy': 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

In [41]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print(AESGCM.generate_key(bit_length=256))

b'2O\xff\xa3.\x85\x8a\xafI\xc1\xbb$k\xd1\x8b4\x07\x13\xdbuN<\x11\x9bm\x0b\xbf\xcf\x92em\xf5'


## rsa_encryption.py

In [42]:
%%writefile ./src/rsa_encryption.py
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import time


def prepare_rsa_key(public_key_pem: str) -> tuple[bytes, bytes, float]:
    start = time.perf_counter()
    session_key = AESGCM.generate_key(bit_length=256)
    end = time.perf_counter()
    
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    aad_component = public_key.encrypt(
        session_key,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return session_key, aad_component, (end-start)

Overwriting ./src/rsa_encryption.py


In [43]:
from src.rsa_encryption import prepare_rsa_key
from src.server_public_key import server_rsa_public_key

print(prepare_rsa_key(server_rsa_public_key))

(b'\xb0\xa3\xa0*O>\x08\x93%\xcc\xa0\xa4N\x96\xab9O\x1d\xbd\xaa\xc6z\x0b\x10\x19\xbd\xf9\xc6\xd0\x19h\xff', b'\xb4s&\xfduSB\x89\xa6\xd9\xee\x06XGX2&\x06a\x84\xd9\xf3\xba\xa2\x89\xd1\x16\x03~\xacd\x0fl\x81\xf0W\xdf\x84\xc1\x01\xb0\xf7\xe8\xe1\xdc\x8ep\xca>y66\x10\xc5\xcb\xd8N\']\xbf\xed\xea{\xe9Y"V\xf87u+\xf8\xd6\xce\x1c\x86\xb9\xf4r\x9f&\xb0 *\x15\xff\x97\x80\xd4\xab\xd7+(3,\xc4\xa54L\xd0\xc4=l\xab\x05\xdf\xd1G\xc1R\xa5p\x18E\x80\xdb\xec\xf5\r!\x11\x0b\x01\x08\xc1\xd4\xc7]\xb5,xCB{4qyp\x84^<\xdfx\xc0\x08\x94\xfc\xb9\x85\xcb\x05\x05\x95.\xc8\xc5q\xa8r\xf13m\xe2\xd7(Wj\xa8\xac\xaeA\xc0\x9a5\x83{\x9c\x91gn\x94\xfe\x8f\xd8E4\xee\xff\xb1\t\t\x90\xbf\xa3m\x991\x88\x9a\xe6\xad\x82\xf9)\xacs\xc3r\x91\x82\x00\x02\x08\x81/\x03})\xc8\x99\xbd\x01\xe6\xe8nM\x86\xe2Yc0\x85\xe8\xae\x1bb\x0c\xd2\xa90@\x91\xb9\xb9?\x17%\x89rR\x16&\xc6\xdf\xd4\xfc', 2.9399991035461426e-05)


## ecc_encryption.py

In [44]:
%%writefile ./src/ecc_encryption.py
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.serialization import load_pem_public_key, Encoding, PublicFormat
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.hashes import SHA256
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import time

def prepare_ecc_key(public_key_pem: str) -> tuple[bytes, bytes, float]:
    server_public = load_pem_public_key(public_key_pem.encode("utf-8"))

    start = time.perf_counter()
    ephemeral_private = X25519PrivateKey.generate()
    

    ephemeral_public  = ephemeral_private.public_key()
    ephemeral_public_bytes = ephemeral_public.public_bytes(Encoding.DER, PublicFormat.SubjectPublicKeyInfo)

    shared_secret = ephemeral_private.exchange(server_public)
    session_key = HKDF(
        algorithm=SHA256(),
        length=32,
        salt=None,
        info=b"sensor-server-v1"
    ).derive(shared_secret)
    end = time.perf_counter()
    return session_key, ephemeral_public_bytes, (end-start)

Overwriting ./src/ecc_encryption.py


## build_payload.py

In [45]:
%%writefile ./src/build_payload.py
from datetime import datetime, timezone, timedelta
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import json
from sensor_simulation import sensor_simulation
import sqlite3
import requests
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey, X25519PublicKey

class EdgeGateway:
	def __init__(self, sensor_id, return_aad_bytes: bool = False):
		self.sensor_id = sensor_id
		self.return_aad_bytes = return_aad_bytes
		self.db_conn = sqlite3.connect(f"sensor{sensor_id}.db")
		cursor_init = self.db_conn.cursor()
		# cursor_init.execute("""
		# 	CREATE TABLE IF NOT EXISTS self_id (
		# 	id INTEGER PRIMARY KEY)
		# """)
		cursor_init.execute("""
			CREATE TABLE IF NOT EXISTS session_keys (
			id			INTEGER	PRIMARY KEY,
			key 		TEXT	NOT NULL,
			times_used	INTEGER	NOT NULL)
		""")
		cursor_init.execute("""
			CREATE TABLE IF NOT EXISTS sensor_readings (
			id			INTEGER	PRIMARY KEY,
			data		TEXT	NOT NULL,
			sent_status	INTEGER	NOT NULL)
		""")
		# cursor_init.execute("INSERT OR IGNORE INTO self_id (id) VALUES (?)", (sensor_id,))
		self.db_conn.commit()

	# def get_latest_session_key(self):
	# 	cursor = self.db_conn.cursor()
	# 	cursor.execute("""
	# 			SELECT id, key FROM session_keys ORDER BY id DESC LIMIT 1
	# 		""")
	# 	return cursor.fetchone()

	def get_latest_session_key(self):
		return self.db_conn.execute("""
				SELECT id, key FROM session_keys ORDER BY id DESC LIMIT 1
			""").fetchone()

	

	def build_payload(self, mode: str, session_key, key) -> dict:
		tz_wib = timezone(timedelta(hours=7))

		# this one gonna cascade depending on the result. None is not a valid session_key. You need to use the current session_key
		# session_key = get_session_key() # -> raw_bytes
		# session_key for testing because get_session_key isn't finished yet
		
		aad = {
			'sensor_id': self.sensor_id,
			'mode': mode,
			'transmission_timestamp': datetime.now(tz_wib).isoformat(),
			'key': base64.b64encode(key).decode('utf-8') 
		}
		aad_bytes = json.dumps(aad, separators=(',', ':'), sort_keys=True).encode('utf-8')

		aesgcm = AESGCM(session_key)
		iv = os.urandom(12)

		plaintext_string = self.db_conn.execute("""
			SELECT data FROM sensor_readings
			WHERE sent_status = 0
			ORDER BY id ASC
			LIMIT 1
		""").fetchone()[0]
		# print(plaintext_string)

		plaintext_bytes = plaintext_string.encode('utf-8') 
		# print(plaintext_bytes)
		encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
		ciphertext = encrypted_raw[:-16]
		tag = encrypted_raw[-16:]

		return {
			'aad': aad if not self.return_aad_bytes else aad_bytes.decode('utf-8'),
			'nonce': base64.b64encode(iv).decode('utf-8'),
			'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
			'tag': base64.b64encode(tag).decode('utf-8')
		}

	def post(self, url: str, *, json: dict):
		response=requests.post(url, json=json)
		if response.status_code == 200:
			self.db_conn.execute("""
				UPDATE sensor_readings
				SET sent_status = 1
				WHERE id = (
					SELECT id
					FROM sensor_readings
					WHERE sent_status = 0
					ORDER BY id ASC
					LIMIT 1
				)
			""")
			self.db_conn.commit()
		return response

	def generate_sensor_data(self, dummy_size_kb: int = 1):
		plaintext_string = json.dumps(sensor_simulation(dummy_size_kb))# -> dict[str,float]

		self.db_conn.execute("""
			INSERT INTO sensor_readings (data, sent_status)
			VALUES (?, 0)
		""", (plaintext_string,))
		self.db_conn.commit()



Overwriting ./src/build_payload.py


## server_public_key.py

In [46]:
%%writefile ./src/server_public_key.py

import os

certs_dir = os.path.join('.', 'certs')

with open(os.path.join(certs_dir, 'rsa_public_key.pem'), 'r', encoding='utf-8') as f:
    server_rsa_public_key = f.read()

with open(os.path.join(certs_dir, 'ecc_public_key.pem'), 'r', encoding='utf-8') as f:
    server_ecc_public_key = f.read()


Overwriting ./src/server_public_key.py


In [47]:
from src.server_public_key import server_rsa_public_key, server_ecc_public_key
print(server_rsa_public_key)
print(server_ecc_public_key)

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAxmXNlw9wR6Uoldfae/yr
BTnHVwZYzKobHcff8rhRjfgJcrsXLBPIk2Gppxk4njEu12anDMuC4btIxXrgu3Lf
oWvZ5HwyAggC/jd1+AhJSZ59OW/ajDQKT6PUQJ8vas4ctb2ayUkvsQ2FOGrJbb+3
mkhmTk3IvqvcJb/dFO9buwk2uM0NFFqSdyXZd/PYncr+/3ok5mdl4M4W1mCA4UCM
QWpuHxk1z5gDIaK2Fs9mGulrvXbU6x9qdjoO5iw1A/soMS9DiF2DWrgWYXD4oXeq
kdlZ2KiMNET69BfY0+/GJgbFrzcScbDEcs9mUURNmhx/F+3NHht5abJkdTh/upjO
vwIDAQAB
-----END PUBLIC KEY-----

-----BEGIN PUBLIC KEY-----
MCowBQYDK2VuAyEA4Nr9fEPc9xLIvemLKV3RvCqlIVJnK6AWceC+ZNT52TE=
-----END PUBLIC KEY-----



## Send payload

In [50]:
import requests
import time
# from build_payload2 import build_payload2
# from get_public_key import get_public_key
import numpy as np
from scipy import stats
import json
import pandas as pd

from src.rsa_encryption import prepare_rsa_key
from src.ecc_encryption import prepare_ecc_key
from src.build_payload import EdgeGateway
from src.server_public_key import server_rsa_public_key, server_ecc_public_key

url = 'http://localhost:3000/telemetry'

sensor1 = EdgeGateway(1)
modes = ['rsa', 'ecc']
sizes = [1, 10, 100]

for mode in modes:
    for size in sizes:
        durations_encryption = []
        packet_size = []
        generate_durations = []
        decryption_durations = []

        for _ in range(30):
            sensor1.generate_sensor_data()

        start_tp = time.perf_counter()
        for _ in range(30):
            start_encrypt = time.perf_counter()
            session_key, key, generate_dur = prepare_rsa_key(server_rsa_public_key) if mode == 'rsa' else prepare_ecc_key(server_ecc_public_key)
            payload = sensor1.build_payload(mode, session_key, key)
            end_encrypt = time.perf_counter()
            
            decryption_duration=sensor1.post(url, json=payload).json()['decryption_duration']
            
            durations_encryption.append(end_encrypt - start_encrypt)
            decryption_durations.append(decryption_duration)
            packet_size.append(len(json.dumps(payload).encode()))
            generate_durations.append(generate_dur)
        end_tp = time.perf_counter()

        df = pd.DataFrame({'Encryption (second)': durations_encryption,
                           'decryption (second)': decryption_durations,
                           'size (byte)': packet_size,
                           'session key generation (second)': generate_durations})
        df.to_csv(f'{mode} {size}kb.csv',index=False)

        df_tp = pd.DataFrame({'time (second)': [start_tp], 'time after 30 messages sent (second)': [end_tp]})
        df_tp.to_csv(f'{mode} {size}kb throughput.csv',index=False)

        print(f'{mode} {size}kb')

        d1 = np.array(durations_encryption) * 1000
        print(f"Encrypt- mean: {np.mean(d1):.3f} ms, median: {np.median(d1):.3f} ms, std: {np.std(d1):.3f} ms")

        d2 = np.array(decryption_durations) * 1000
        print(f"Decrypt- mean: {np.mean(d2):.3f} ms, median: {np.median(d2):.3f} ms, std: {np.std(d2):.3f} ms")

        print(f"Throughput: {30/(end_tp-start_tp)} message/second")

        d3 = np.array(packet_size)
        print(f"JSON size - mean: {np.mean(d3):.3f} bytes, median: {np.median(d3):.3f} bytes, std: {np.std(d3):.3f} bytes")

        d4 = np.array(generate_durations) * 1000
        print(f"Generate key- mean: {np.mean(d4):.3f} ms, median: {np.median(d4):.3f} ms, std: {np.std(d4):.3f} ms")

        print('='*80+'\n')

rsa 1kb
Encrypt- mean: 2.165 ms, median: 0.794 ms, std: 7.380 ms
Decrypt- mean: 2441.113 ms, median: 2292.050 ms, std: 795.432 ms
Throughput: 28.750475581755182 message/second
JSON size - mean: 2179.667 bytes, median: 2179.000 bytes, std: 1.491 bytes
Generate key- mean: 0.415 ms, median: 0.023 ms, std: 2.105 ms

rsa 10kb
Encrypt- mean: 0.814 ms, median: 0.744 ms, std: 0.139 ms
Decrypt- mean: 2111.130 ms, median: 2107.150 ms, std: 189.394 ms
Throughput: 33.21654764610102 message/second
JSON size - mean: 2180.067 bytes, median: 2179.000 bytes, std: 1.769 bytes
Generate key- mean: 0.025 ms, median: 0.024 ms, std: 0.005 ms

rsa 100kb
Encrypt- mean: 0.903 ms, median: 0.852 ms, std: 0.186 ms
Decrypt- mean: 2326.957 ms, median: 2318.300 ms, std: 264.418 ms
Throughput: 31.242911764674275 message/second
JSON size - mean: 2179.533 bytes, median: 2179.000 bytes, std: 1.360 bytes
Generate key- mean: 0.028 ms, median: 0.025 ms, std: 0.008 ms

ecc 1kb
Encrypt- mean: 1.046 ms, median: 0.855 ms, std: 

# Stop

In [119]:
sensor1.db_conn.execute(
    """DROP TABLE IF EXISTS sensor_readings"""
)
sensor1.db_conn.commit()